# G-Reasoner on v16 soft+canon(0.85) — CARGO **fusion** (operator ⊕ graph, all learned jointly)

Same v16sc G-Reasoner as the base run, but the document score is **fused** with the operator's
anti-hub score by a learned per-query gate — `fused = z(S_op) + γ_q·ReLU(z(graph))`. In ONE run the
loss trains: the **GNN** (from scratch), the **gate** (when to trust the graph), AND the **operator's**
own scalars `w=[w0,w1,w2]` and `β` — the operator is recomputed live from cached raw ingredients
(`operator_components.npz`: dense/S/M, aligned to the v16sc document-node order), so its weights
carry gradients. The BGE encoder stays frozen; `w,β` start at the fitted values and move at ~0.2× LR.


## 1. GPU + Drive

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import os
from google.colab import drive
drive.mount('/content/drive')
DRIVE = "/content/drive/MyDrive/cargo-gfmrag"   # holds gfm-rag-adapted.zip + the v16sc bundle
# HF token (helps rate limits; tokenizer.json fetch below wants the auth header)
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    os.environ.setdefault("HF_TOKEN", "")   # public model still works without it

## 2. Install the adapted G-Reasoner engine (same as v15/v6rel runs)

In [ ]:
import os, sys, torch
!rm -rf /content/gfm-rag
!cd /content && unzip -q {DRIVE}/gfm-rag-adapted.zip
!pip install -q --no-deps -e /content/gfm-rag
# TORCH IS DELIBERATELY NOT IN THIS LIST. Colab ships a torch/torchvision pair built
# against each other, and asking pip for `torch` can move torch off the version its
# torchvision was compiled for. THAT mismatch is what produced
#   ImportError: cannot import name 'VideoReader'
# from datasets' torch formatter. gfmrag installs --no-deps, so nothing here needs a
# torch newer than the host image's.
!pip install -q torch-geometric sentence-transformers transformers hydra-core omegaconf \
              easydict ninja faiss-cpu pymetis wandb tqdm numpy pandas python-dotenv \
              langchain-community 2>&1 | tail -3

# REPAIR, NEVER REMOVE. This used to be `pip uninstall -y torchvision`, on the reasoning
# that gfmrag does not use it. That reasoning has expired: current transformers resolves
# PreTrainedModel through a lazy module that imports torchvision, so deleting it turns
# every `from transformers import ...` into
#   ModuleNotFoundError: Could not import module 'PreTrainedModel'
# and takes sentence-transformers, and therefore every encoder in this notebook, down with
# it. Verify the pair instead, and only intervene if it is actually broken.
import torch
try:
    import torchvision
    from transformers import PreTrainedModel          # the import that has to work
    print(f"torch {torch.__version__} | torchvision {torchvision.__version__} | transformers ok")
except Exception as _e:
    # STOP, DO NOT SELF-HEAL. The obvious repair, `pip install torchvision`, resolves to
    # the LATEST torchvision and drags torch up with it: measured on 2026-08-17 it took a
    # stock runtime from torch 2.11.0+cu128 to 2.13.0+cu130, a 2 GB download that rebuilt
    # the whole CUDA stack, broke Colab's cudf/cuml/raft pins, and left the running kernel
    # holding the OLD torch. Silently re-pinning CUDA under a training run is far worse
    # than refusing, because the damage only surfaces as `torch.cuda.is_available()` going
    # False, or as numerics nobody can reproduce.
    #
    # A matched pair is what the stock image already ships. Getting back to it is one
    # menu action, and no pip incantation is more reliable than that.
    raise RuntimeError(
        f"torchvision/transformers are broken in this runtime "
        f"({type(_e).__name__}: {_e}).\n"
        f"torch here is {torch.__version__}.\n"
        f"FIX: Runtime > Disconnect and DELETE runtime (not 'Restart session' -- a restart "
        f"keeps whatever pip did), then run this notebook from the top. The stock image "
        f"ships a matched torch/torchvision pair and this cell installs neither, so the "
        f"check above will pass with no downloads.\n"
        f"Do NOT `pip install torchvision` to get past this: it upgrades torch and CUDA "
        f"underneath you."
    ) from _e

# THE OTHER HALF OF THE TORCHVISION STORY. datasets' torch formatter runs
# `from torchvision.io import VideoReader` whenever torchvision is importable, and
# current torchvision has REMOVED VideoReader. That import sits on the training
# dataloader's hot path, so with torchvision present every training run dies on its
# first batch. Uninstalling torchvision was the old workaround and now breaks
# transformers instead (see above), so the remaining move is to patch datasets ON
# DISK: guard the import, skip the isinstance when it is unavailable. The training
# subprocess re-imports datasets from disk, so the patch reaches it with no restart.
import re as _re
import datasets.formatting.torch_formatter as _dtf
_p = _dtf.__file__
_s = open(_p).read()
if "VideoReader = None" in _s:
    print("datasets torch formatter already patched")
else:
    _s2 = _re.sub(r'^( *)from torchvision\.io import VideoReader$',
                  lambda m: (f"{m.group(1)}try:\n{m.group(1)}    from torchvision.io import VideoReader\n"
                             f"{m.group(1)}except Exception:\n{m.group(1)}    VideoReader = None"),
                  _s, flags=_re.M)
    _s2 = _s2.replace("isinstance(value, VideoReader)",
                      "(VideoReader is not None and isinstance(value, VideoReader))")
    assert _s2 != _s, "VideoReader import not found -- datasets layout changed, patch by hand"
    open(_p, "w").write(_s2)
    print("patched datasets torch formatter:", _p)
# Replay the exact failing path in a fresh interpreter: torchvision imported (that is
# what arms the buggy branch), then a torch-formatted Dataset read.
import subprocess as _sp
_r = _sp.run([sys.executable, "-c",
              "import torchvision, datasets\n"
              "d = datasets.Dataset.from_dict({'x': [1, 2, 3]}).with_format('torch')\n"
              "print('datasets formatter ok:', d[:2]['x'])"],
             capture_output=True, text=True)
print(_r.stdout.strip())
assert _r.returncode == 0, _r.stderr[-2000:]

def soft_import(path, line, fallback):
    t = open(path).read()
    if f"try:\n    {line}" not in t:
        open(path, "w").write(t.replace(line, f"try:\n    {line}\nexcept Exception:\n    {fallback}"))
soft_import("/content/gfm-rag/gfmrag/text_emb_models/__init__.py",
            "from .qwen3_model import Qwen3TextEmbModel", "Qwen3TextEmbModel = None")
# pylate/ColBERT entity-linker is unused by SFT training; make its import non-fatal so the
# training subprocess (fresh Python re-imports gfmrag) does not crash on `import pylate`.
soft_import("/content/gfm-rag/gfmrag/graph_index_construction/entity_linking_model/__init__.py",
            "from .colbert_el_model import ColbertELModel", "ColbertELModel = None")
# 4c. LLM-OpenIE model imports langchain_community (ChatOllama/ChatLlamaCpp); the installed version
#     dropped ChatOllama. SFT training never builds an index, so make this import non-fatal.
soft_import("/content/gfm-rag/gfmrag/graph_index_construction/openie_model/__init__.py",
            "from .llm_openie_model import LLMOPENIEModel", "LLMOPENIEModel = None")
# 4d. same for the LLM-NER model (separate __init__, separate import line).
soft_import("/content/gfm-rag/gfmrag/graph_index_construction/ner_model/__init__.py",
            "from .llm_ner_model import LLMNERModel", "LLMNERModel = None")
# the adapted zip strips config/wandb/ (and sometimes config/text_emb_model/) -> create before writing
for _cd in ["/content/gfm-rag/gfmrag/workflow/config/wandb",
            "/content/gfm-rag/gfmrag/workflow/config/text_emb_model"]:
    os.makedirs(_cd, exist_ok=True)
open("/content/gfm-rag/gfmrag/workflow/config/wandb/default.yaml", "w").write(
    'enabled: false\nlog_model: false\nproject: "gfm-rag"\nentity: null\nname: null\ngroup: null\ntags: []\nnotes: ""\n')
open("/content/gfm-rag/gfmrag/workflow/config/text_emb_model/qwen3_st.yaml", "w").write(
    '_target_: gfmrag.text_emb_models.BaseTextEmbModel\n'
    'text_emb_model_name: /content/qwen3\nnormalize: True\nbatch_size: 32\n'   # LOCAL path, not hub name (see cell 3)
    'query_instruct: "Instruct: Given a scientific research problem or open need, retrieve papers whose method, mechanism, or technique could be borrowed as inspiration, including transfers from other domains.\\nQuery: "\n'
    'passage_instruct: null\nmodel_kwargs: null\n')
sys.path.insert(0, "/content/gfm-rag")
# stub the unused pylate/ColBERT dep so the IN-KERNEL gfmrag import does not crash
import types as _t
for _m in ["pylate", "pylate.indexes", "pylate.models", "pylate.retrieve"]:
    sys.modules.setdefault(_m, _t.ModuleType(_m))
class _D:
    def __init__(self, *a, **k): pass
sys.modules["pylate.indexes"].PLAID = _D; sys.modules["pylate.models"].ColBERT = _D; sys.modules["pylate.retrieve"].ColBERT = _D
from gfmrag.models.gfm_reasoner import GraphReasoner
assert torch.cuda.is_available(), "Use an A100/high-RAM GPU"
print("G-Reasoner OK |", torch.cuda.get_device_name(0))

# ---- PATCH: per-epoch STRATIFIED metrics ----
# Training runs in a subprocess, so we edit the source: wrap trainer.evaluate() to add
# per-slice document_hits@k/mrr keys. _log_metrics then prints them EACH EPOCH in the same
# format as the aggregate lines. Toggle with STRAT_EVAL=0; BGE split from STRAT_BGE.
STF = "/content/gfm-rag/gfmrag/workflow/sft_training.py"
_src = open(STF).read()
if "_evaluate_stratified" not in _src:
    _inject = "\n".join([
        "    # --- injected: per-epoch stratified eval (ZERO extra forward pass) ---",
        "    # A forward hook records each eval query's gold-document rank during evaluate()'s",
        "    # existing pass; we then add per-slice keys to the metrics dict (logged as usual).",
        "    import os as _o, json as _j",
        "    from collections import defaultdict as _dd",
        "    _rec = []",
        "    _recording = {'on': False}",
        "    def _hook(_module, _inp, _out):",
        "        if not _recording['on'] or len(_inp) < 2:",
        "            return",
        "        try:",
        "            _g, _b = _inp[0], _inp[1]",
        "            _did = _g.nodes_by_type['document']",
        "            _dp = _out[:, _did]",
        "            _tgt = _b['target_nodes_mask'][:, _did].bool()",
        "            _rk = _dp.argsort(dim=-1, descending=True).argsort(dim=-1)",
        "            _ids = _b['id']",
        "            for _qi in range(_dp.shape[0]):",
        "                _pos = _tgt[_qi].nonzero(as_tuple=True)[0]",
        "                if len(_pos):",
        "                    _r = int(_rk[_qi, _pos].min().item()) + 1",
        "                    _q = _ids[_qi]",
        "                    _q = _q.item() if hasattr(_q, 'item') else _q",
        "                    _rec.append((_q, _r))",
        "        except Exception:",
        "            pass",
        "    trainer.model.register_forward_hook(_hook)",
        "    _orig_evaluate = trainer.evaluate",
        "    def _evaluate_stratified():",
        "        _rec.clear(); _recording['on'] = True",
        "        m = _orig_evaluate()",
        "        _recording['on'] = False",
        "        if _o.environ.get('STRAT_EVAL','1') != '1':",
        "            return m",
        "        try:",
        "            _name = _o.environ.get('STRAT_NAME','eval')",
        "            _tj = _o.environ.get('STRAT_TEST','')",
        "            _bp = _o.environ.get('STRAT_BGE','')",
        "            _meta = {q['id']: q for q in _j.load(open(_tj))} if _tj and _o.path.exists(_tj) else {}",
        "            _BGE = {r['id']: r for r in _j.load(open(_bp))} if _bp and _o.path.exists(_bp) else {}",
        "            def _brank(qid, g):",
        "                for i,(d,_s) in enumerate(_BGE.get(qid,{}).get('predictions',{}).get('document',[]),1):",
        "                    if d==g: return i",
        "                return 10**9",
        "            _sl = _dd(lambda: _dd(list)); _seen = set()",
        "            for _q,_r in _rec:",
        "                if _q in _seen: continue",
        "                _seen.add(_q)",
        "                _mq = _meta.get(_q, {})",
        "                _gg = _mq.get('supporting_documents') or []",
        "                _gd = _gg[0] if isinstance(_gg,list) and _gg else _gg",
        "                _st = 'same' if _mq.get('stratum')=='same' else 'cross'",
        "                _sim = 'dissim' if _brank(_q,_gd)>100 else 'sim'",
        "                for _nm in [_st,_sim] + (['cross+dissim'] if (_st=='cross' and _sim=='dissim') else []):",
        "                    _d = _sl[_nm]",
        "                    for _k in (1,5,10): _d['hits@'+str(_k)].append(float(_r<=_k))",
        "                    _d['mrr'].append(1.0/_r if _r<=100 else 0.0)",
        "            for _nm,_d in _sl.items():",
        "                _n = len(_d['mrr']) or 1",
        "                for _c in ('hits@1','hits@5','hits@10','mrr'):",
        "                    m[_name+'/document_'+_c+'/'+_nm] = sum(_d[_c])/_n",
        "        except Exception as _e:",
        "            print('[stratified] skipped:', _e)",
        "        return m",
        "    trainer.evaluate = _evaluate_stratified",
        "    trainer.train()",
    ])
    _src = _src.replace("    trainer.train()", _inject, 1)
    open(STF, "w").write(_src)
    print("patched sft_training.py -> per-epoch stratified eval")
else:
    print("stratified patch already applied")
# ---- PATCH: tqdm shows per-component RUNNING-AVERAGE losses (bce / pcr / mse / total) ----
# Patches base_trainer.py ON DISK so the training SUBPROCESS shows every loss, not just the total.
_bt = "/content/gfm-rag/gfmrag/trainers/base_trainer.py"
_bs = open(_bt).read()
_OLD = 'progress_bar.set_postfix(loss=step_metrics.get("loss", 0.0))'
_NEW = ('_names = {"bce_loss": "bce", "pcr_loss": "pcr", "mse_loss": "mse", "loss": "tot"}\n'
        '                progress_bar.set_postfix({_names.get(k, k): f"{np.mean(v):.3f}" for k, v in epoch_metrics.items()})')
if "_names.get(k, k)" in _bs:
    print("base_trainer.py already patched (per-component postfix present)")
elif _OLD in _bs:
    open(_bt, "w").write(_bs.replace(_OLD, _NEW))
    print("patched base_trainer.py -> tqdm shows running-average bce/pcr/mse/tot")
else:
    print("WARN: postfix line not found in base_trainer.py (file may have changed); inspect ~line 424")


In [ ]:
# === write the CARGO-fusion files into the fork (self-contained; runs after the engine install) ===
# Objective = OPERATOR-HARD-NEGATIVE contrastive (report Eq 3.9): negatives are the operator's own
# top hubs, so the graph is forced to fix the operator's misses. Gate gamma_init=0.01 (opt-in).
import json, os
FILES = json.loads(r'''{"/content/gfm-rag/gfmrag/models/fusion_reasoner.py": "\"\"\"\nfusion_reasoner.py \u2014 CARGO fusion: v16sc G-Reasoner + operator, EVERYTHING learned jointly in one run.\n\nWraps the standard GraphReasoner. The operator is recomputed LIVE from its raw ingredients so its\nweights and exponent are trainable (matches the interim report: beta and the fusion weights are learned):\n\n    S_op      = w0 z(dense) + w1 z(S / dem^beta) + w2 z(M / dem^beta)   # dem = total_S - S (anti-hub)\n    fused_doc = z(S_op) + gamma_q * relu( z(graph_doc) )\n    gamma_q   = softplus( gate(coverage) )                             # per-query gate >= 0\n\nTrained end-to-end from one ranking loss on fused_doc:\n  - the GNN weights (the v16sc graph reasoner, from scratch),\n  - the gate (when to trust the graph), and\n  - the operator scalars w=[w0,w1,w2] and beta.\nThe BGE encoder that produced dense/S/M is frozen (we only learn the handful of combination scalars).\nw, beta are warm-started at the fitted values and learn at the base LR (op_lr_scale=1.0) \u2014 4 params on\na near-convex objective, so they converge fast. relu => the graph can only promote a doc (aggregate floor).\n\nIngredients come from env OPERATOR_COMPONENTS (train) / OPERATOR_COMPONENTS_TEST (test): npz with\n{dense,S,M: float16 [Q x n_doc] in nodes.csv doc order, total_S: float32 [n_doc], query_ids:[...]}.\nQuery ids are split-unique, so both tables are merged and looked up by batch['id'].\n\"\"\"\nimport math\nimport os\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F  # noqa: N812\n\nfrom gfmrag.models.gfm_reasoner import GraphReasoner\n\nW_INIT = (1.05, 1.05, 0.25)   # fitted operator fusion weights\nBETA_INIT = 0.95              # fitted anti-hub exponent\n\n\nclass FusionGraphReasoner(nn.Module):\n    def __init__(self, entity_model, feat_dim, gamma_init=0.5, gate_hidden=8,\n                 op_lr_scale=1.0, **kwargs):\n        super().__init__()\n        self.base = GraphReasoner(entity_model, feat_dim, **kwargs)\n\n        # --- per-query gate: gamma_q = softplus(gate(operator coverage)) ---\n        self.gate = nn.Sequential(\n            nn.Linear(1, gate_hidden), nn.ReLU(), nn.Linear(gate_hidden, 1)\n        )\n        nn.init.zeros_(self.gate[-1].weight)\n        nn.init.constant_(self.gate[-1].bias, math.log(math.expm1(max(gamma_init, 1e-3))))\n\n        # --- trainable operator scalars (warm-started at fitted values; base LR via op_lr_scale) ---\n        # effective value = init + op_lr_scale * delta, delta starts at 0. With Adam this makes the\n        # operator learn at op_lr_scale x the graph/gate LR (1.0 = same rate).\n        self.op_lr_scale = float(op_lr_scale)\n        self.register_buffer(\"w_init\", torch.tensor(W_INIT, dtype=torch.float32))\n        self.w_delta = nn.Parameter(torch.zeros(3))\n        self.beta_delta = nn.Parameter(torch.zeros(()))\n\n        # --- operator raw ingredients (CPU float16; rows moved to GPU per batch) ---\n        self._dense: dict[str, torch.Tensor] = {}\n        self._S: dict[str, torch.Tensor] = {}\n        self._M: dict[str, torch.Tensor] = {}\n        self._totS: dict[str, torch.Tensor] = {}\n        self._row: dict[str, tuple] = {}\n        for tag, ev in ((\"train\", \"OPERATOR_COMPONENTS\"), (\"test\", \"OPERATOR_COMPONENTS_TEST\")):\n            p = os.environ.get(ev)\n            if not p:\n                continue\n            d = np.load(p, allow_pickle=True)\n            self._dense[tag] = torch.from_numpy(np.asarray(d[\"dense\"], dtype=np.float16))\n            self._S[tag] = torch.from_numpy(np.asarray(d[\"S\"], dtype=np.float16))\n            self._M[tag] = torch.from_numpy(np.asarray(d[\"M\"], dtype=np.float16))\n            self._totS[tag] = torch.from_numpy(np.asarray(d[\"total_S\"], dtype=np.float32))\n            for i, q in enumerate(d[\"query_ids\"]):\n                self._row[str(q)] = (tag, i)\n            print(f\"[fusion] loaded {ev}: dense/S/M {tuple(self._dense[tag].shape)} ({len(d['query_ids'])} queries)\")\n        assert self._row, \"no operator components: set OPERATOR_COMPONENTS / OPERATOR_COMPONENTS_TEST\"\n\n        self._raw_doc = None   # graph-alone doc scores, cached each forward for the aux loss\n        self._doc_ids = None\n        self._s_op = None      # live operator scores [B, n_doc] (detached), cached for hard-neg mining\n\n    @staticmethod\n    def _z(x):\n        return (x - x.mean(-1, keepdim=True)) / (x.std(-1, keepdim=True) + 1e-6)\n\n    def _operator(self, ids, n_doc, device):\n        \"\"\"Recompute S_op live from cached ingredients with the current (trainable) w, beta.\"\"\"\n        order = [self._row[str(x.item() if hasattr(x, \"item\") else x)] for x in ids]\n        tag = order[0][0]\n        assert all(t == tag for t, _ in order), \"batch mixes train/test operator tables\"\n        idx = torch.tensor([r for _, r in order], dtype=torch.long)\n\n        dense = self._dense[tag].index_select(0, idx).to(device, torch.float32)\n        S = self._S[tag].index_select(0, idx).to(device, torch.float32)\n        M = self._M[tag].index_select(0, idx).to(device, torch.float32)\n        totS = self._totS[tag].to(device)                       # [n_doc]\n        assert dense.shape[1] == n_doc, f\"operator cols {dense.shape[1]} != {n_doc} doc nodes (alignment)\"\n\n        w = self.w_init + self.op_lr_scale * self.w_delta       # [3]\n        beta = (BETA_INIT + self.op_lr_scale * self.beta_delta).clamp(min=0.05)\n        dem = (totS.unsqueeze(0) - S).clamp(min=1e-6)           # [B, n_doc] leave-one-out popularity\n        degb = dem.pow(beta)\n        return w[0] * self._z(dense) + w[1] * self._z(S / degb) + w[2] * self._z(M / degb)\n\n    def forward(self, graph, batch, entities_weight=None):\n        g = self.base(graph, batch, entities_weight)            # [B, N] per-node scores\n        doc = graph.nodes_by_type[\"document\"].to(g.device)      # nodes.csv document order\n        gdoc = g.index_select(1, doc).float()                   # [B, n_doc] graph-alone doc scores\n\n        s_op = self._operator(batch[\"id\"], doc.numel(), g.device)   # [B, n_doc] live operator\n        cov = s_op.topk(min(5, s_op.shape[1]), dim=-1).values.mean(-1, keepdim=True)  # coverage\n        gamma = F.softplus(self.gate(cov)).squeeze(-1)          # [B] per-query gate\n        fused = self._z(s_op) + gamma[:, None] * torch.relu(self._z(gdoc))\n\n        out = g.clone()\n        out[:, doc] = fused.to(out.dtype)\n        self._raw_doc = gdoc          # cache for the graph-alone auxiliary loss (train only)\n        self._doc_ids = doc\n        self._s_op = s_op.detach()    # cache for operator-hard-negative mining (train only)\n        return out\n", "/content/gfm-rag/gfmrag/trainers/fusion_trainer.py": "\"\"\"\nfusion_trainer.py \u2014 SFTTrainer for the CARGO fusion (FusionGraphReasoner).\n\nTwo objectives, selected by env FUSION_OBJECTIVE:\n\n- \"bce_pcr\" (default, legacy): the config losses (bce + pcr) act on the FUSED document scores, plus a\n  small graph-alone ListCE aux (weight AUX_W). This is the run whose graph stayed inert (aux frozen at\n  ~log N): the operator already satisfies bce/pcr on the easy queries, so the GNN never gets a gradient.\n\n- \"hardneg\" (the report's Eq 3.9 objective): a softmax cross-entropy over a per-query lineup whose\n  negatives are the OPERATOR'S OWN top-ranked hubs \u2014 {gold} u operator-top-`HARDNEG_HUB` u `HARDNEG_RAND`\n  random docs. Ranking the gold above the docs the operator already loves can only be done with the graph,\n  so this is what actually teaches the graph to fix the operator's cross-domain misses. It is applied to\n  BOTH the fused score (trains gate + operator scalars + graph jointly) AND the graph-alone score\n  (weight AUX_W \u2014 trains the GNN DIRECTLY, so it still learns even when the gate gamma_q is initialised\n  near zero and the fused-path gradient into the graph is tiny).\n\nOnly train_step is overridden; evaluate()/predict() are inherited unchanged and already consume the\nfused score, because the fusion lives inside the model's forward().\n\"\"\"\nimport os\n\nimport torch\n\nfrom gfmrag.losses import ListCELoss\nfrom gfmrag.models.ultra import query_utils\n\nfrom .sft_trainer import SFTTrainer\n\n\nclass FusionSFTTrainer(SFTTrainer):\n    def __init__(self, *args, **kwargs):\n        super().__init__(*args, **kwargs)\n        self._aux_loss_fn = ListCELoss()\n        self._aux_w = float(os.environ.get(\"AUX_W\", \"0.1\"))\n        self._objective = os.environ.get(\"FUSION_OBJECTIVE\", \"bce_pcr\")\n        self._hn_hub = int(os.environ.get(\"HARDNEG_HUB\", \"50\"))    # operator-top-K hubs per query\n        self._hn_rand = int(os.environ.get(\"HARDNEG_RAND\", \"50\"))  # random negatives per query\n\n    def _contrastive_hardneg(self, doc_scores, target_doc, s_op):\n        \"\"\"Multi-positive InfoNCE over {gold} u operator-top-K hubs u random negatives.\n\n        doc_scores : [B, n_doc] scores to train (fused or graph-alone), require grad.\n        target_doc : [B, n_doc] {0,1} gold mask over document nodes.\n        s_op       : [B, n_doc] operator scores (detached) \u2014 mines the hard-negative hubs.\n        \"\"\"\n        B, n_doc = doc_scores.shape\n        dev = doc_scores.device\n        loss = doc_scores.new_zeros(())\n        n = 0\n        for b in range(B):\n            pos = target_doc[b].nonzero(as_tuple=True)[0]\n            if pos.numel() == 0:\n                continue\n            # operator's own top hubs (the tempting distractors), excluding the gold(s)\n            k = min(self._hn_hub + pos.numel(), n_doc)\n            hub = s_op[b].topk(k).indices\n            hub = hub[~torch.isin(hub, pos)][: self._hn_hub]\n            # random negatives (background)\n            rand = torch.randint(0, n_doc, (self._hn_rand,), device=dev)\n            rand = rand[~torch.isin(rand, pos)]\n            cand = torch.cat([pos, hub, rand])\n            logits = doc_scores[b, cand]\n            log_z = torch.logsumexp(logits, 0)\n            log_pos = torch.logsumexp(logits[: pos.numel()], 0)\n            loss = loss + (log_z - log_pos)\n            n += 1\n        return loss / max(n, 1)\n\n    def train_step(self, batch, task_dataset):\n        graph = task_dataset.graph.to(self.device)\n        batch = query_utils.cuda(batch, device=self.device)\n\n        pred = self.parallel_model(graph, batch)          # FUSED [B, N]\n        target = batch[\"target_nodes_mask\"]\n\n        total = torch.tensor(0.0, device=self.device, requires_grad=True)\n        step_metrics = {}\n\n        if self._objective == \"hardneg\":\n            did = self.model._doc_ids\n            s_op = self.model._s_op                        # [B, n_doc] detached operator scores\n            tgt_doc = target[:, did]\n            # (1) fused-score contrastive: trains gate + operator scalars + graph jointly\n            l_fused = self._contrastive_hardneg(pred[:, did], tgt_doc, s_op)\n            step_metrics[\"hn_fused\"] = l_fused.item()\n            total = total + l_fused\n            # (2) graph-alone contrastive: trains the GNN DIRECTLY (survives a near-zero gate)\n            if self._aux_w > 0 and getattr(self.model, \"_raw_doc\", None) is not None:\n                l_graph = self._contrastive_hardneg(self.model._raw_doc, tgt_doc, s_op)\n                step_metrics[\"hn_graph\"] = l_graph.item()\n                total = total + self._aux_w * l_graph\n        else:\n            for sft_loss in self.loss_functions:\n                tids = graph.nodes_by_type[sft_loss.target_node_type]\n                loss = sft_loss.loss_fn(pred[:, tids], target[:, tids])\n                step_metrics[sft_loss.name] = loss.item()\n                total = total + sft_loss.weight * loss\n\n            # graph-alone auxiliary term (teach the GNN independently of the gate)\n            if self._aux_w > 0 and getattr(self.model, \"_raw_doc\", None) is not None:\n                did = self.model._doc_ids\n                aux = self._aux_loss_fn(self.model._raw_doc, target[:, did])\n                step_metrics[\"aux_graph\"] = aux.item()\n                total = total + self._aux_w * aux\n\n        step_metrics[\"loss\"] = total\n        return step_metrics\n", "/content/gfm-rag/gfmrag/workflow/config/gfm_reasoner/sft_training_fusion.yaml": "# G-Reasoner SFT \u2014 CARGO fusion (v16sc graph + operator, learned jointly in ONE run).\n# = base sft_training.yaml, but: (1) model is FusionGraphReasoner (wraps GraphReasoner, fuses the\n# operator score into the document logits with a learned per-query gate); (2) losses are bce + pcr on\n# the FUSED score (NO mse distillation \u2014 the operator is a fused signal, not a teacher); (3) the\n# trainer adds a small graph-alone aux loss (weight AUX_W) so the from-scratch GNN learns alongside.\n# Operator scores come from env OPERATOR_SCORES (train) / OPERATOR_SCORES_TEST (test).\nhydra:\n  run:\n    dir: outputs/qa_finetune/${now:%Y-%m-%d}/${now:%H-%M-%S}\n  searchpath:\n    - pkg://gfmrag.workflow.config\n\ndefaults:\n  - _self_\n  - text_emb_model: qwen3\n  - wandb: default\n\nseed: 1024\ntimeout: 60\nsave_pretrained: no\nload_model_from_pretrained: null\n\ndatasets:\n  _target_: gfmrag.graph_index_datasets.GraphIndexDataset\n  cfgs:\n    root: ./data\n    force_reload: False\n    text_emb_model_cfgs: ${text_emb_model}\n  train_names:\n    - tomato_train_v16sc\n  valid_names:\n    - tomato_test_v16sc\n  init_datasets: True\n  feat_dim: 1024\n  max_datasets_in_memory: 10\n  data_loading_workers: 4\n\n# Fusion model: wraps the v16sc GraphReasoner; entity_model is instantiated and passed through.\n# The operator is recomputed live from cached ingredients (OPERATOR_COMPONENTS[_TEST]) so its\n# scalars w=[w0,w1,w2] and beta are LEARNED jointly (warm-started at fitted values, base LR).\nmodel:\n  _target_: gfmrag.models.fusion_reasoner.FusionGraphReasoner\n  gamma_init: 0.01         # gate near zero: on step 1 the model IS the operator (opt-in graph, protects same-domain)\n  gate_hidden: 8\n  op_lr_scale: 1.0         # operator scalars learn at the full base LR (4 params, warm-started, fast to fit)\n  use_ent_emb: early-late-fusion\n  dtype: bfloat16\n  entity_model:\n    _target_: gfmrag.models.ultra.models.QueryNBFNet\n    input_dim: 1024\n    hidden_dims: [1024, 1024, 1024, 1024, 1024, 1024]\n    message_func: distmult\n    aggregate_func: sum\n    short_cut: yes\n    layer_norm: yes\n    return_hidden: True\n\n# Loss: gold supervision on the FUSED document score (no distillation term).\nlosses:\n  - name: bce_loss\n    loss:\n      _target_: gfmrag.losses.BCELoss\n      adversarial_temperature: 0.2\n    weight: 0.3\n    target_node_type: document\n  - name: pcr_loss\n    loss:\n      _target_: gfmrag.losses.ListCELoss\n    weight: 0.7\n    target_node_type: document\n\noptimizer:\n  _target_: torch.optim.AdamW\n  lr: 5.0e-4\n\ntrainer:\n  _target_: gfmrag.trainers.fusion_trainer.FusionSFTTrainer   # adds graph-alone aux loss (AUX_W)\n  args:\n    _target_: gfmrag.trainers.TrainingArguments\n    train_batch_size: 4\n    num_epoch: 20\n    logging_steps: 100\n    max_steps_per_epoch: null\n    resume_from_checkpoint: null\n    do_train: true\n    do_eval: true\n    save_best_only: yes\n    metric_for_best_model: document_mrr\n    dtype: ${model.dtype}\n    split_graph_inference: false\n    split_graph_training: false\n    split_graph_partition: contiguous\n  metrics: [mrr, hits@1, hits@2, hits@3, hits@5, hits@10, hits@20, recall@2, recall@3, recall@5, recall@10, recall@20]\n  target_types: [document]\n"}''')
for p, c in FILES.items():
    os.makedirs(os.path.dirname(p), exist_ok=True)
    open(p, 'w').write(c)
    print('wrote', p, f'({len(c)} bytes)')
import importlib, sys
sys.path.insert(0, '/content/gfm-rag')
for m in ['gfmrag.models.fusion_reasoner', 'gfmrag.trainers.fusion_trainer']:
    importlib.import_module(m); print('import OK:', m)
from gfmrag.models.fusion_reasoner import FusionGraphReasoner
from gfmrag.trainers.fusion_trainer import FusionSFTTrainer
print('fusion files ready')


## 3. Fetch Qwen3-Embedding-0.6B robustly (HF Xet CDN workaround)

HF migrated files >~10 MB to the **Xet** backend, whose CDN silently stalls / 403s on Colab VMs
(`huggingface_hub`, `snapshot_download`, `hf_transfer`, `HF_HUB_DISABLE_XET` all fail). Fix: pull the
big weight with **aria2c** (multi-connection retries land on the working CDN), small files with plain
**curl** (they bypass Xet), and `tokenizer.json` (11 MB, also Xet) with a **curl retry loop**. Then load
by **local path** so nothing re-triggers a Xet download. Cached to Drive so this is one-time.

In [ ]:
import os, shutil
QDIR = "/content/qwen3"
DRIVE_QWEN = f"{DRIVE}/qwen3-embedding-0.6b"
BASE = "https://huggingface.co/Qwen/Qwen3-Embedding-0.6B/resolve/main"
TOK = os.environ.get("HF_TOKEN", "")
AUTH = f'-H "Authorization: Bearer {TOK}"' if TOK else ""
NEED = ["model.safetensors","config.json","config_sentence_transformers.json","modules.json",
        "tokenizer.json","tokenizer_config.json","vocab.json","merges.txt","1_Pooling/config.json"]

def ready(d):
    return (all(os.path.exists(f"{d}/{f}") for f in NEED)
            and os.path.getsize(f"{d}/model.safetensors") > 1_000_000_000
            and os.path.getsize(f"{d}/tokenizer.json") > 11_000_000)

if not ready(QDIR) and ready(DRIVE_QWEN):
    print("restoring Qwen3 from Drive cache ..."); shutil.copytree(DRIVE_QWEN, QDIR, dirs_exist_ok=True)

if not ready(QDIR):
    os.makedirs(f"{QDIR}/1_Pooling", exist_ok=True)
    # 1) big weight via aria2c (run ONCE — a repeat can delete the finished file)
    if not (os.path.exists(f"{QDIR}/model.safetensors") and os.path.getsize(f"{QDIR}/model.safetensors") > 1_000_000_000):
        os.system("apt-get -qq install -y aria2")
        os.system(f'aria2c -x16 -s16 -k1M --max-tries=5 --retry-wait=2 --file-allocation=none '
                  f'{AUTH} -d {QDIR} -o model.safetensors "{BASE}/model.safetensors"')
    # 2) small files bypass Xet — plain curl
    for f in ["config.json","config_sentence_transformers.json","modules.json",
              "tokenizer_config.json","vocab.json","merges.txt","1_Pooling/config.json"]:
        os.system(f'curl -sSL -f {AUTH} "{BASE}/{f}" -o "{QDIR}/{f}"')
    # 3) tokenizer.json (11 MB, also Xet) — retry until a request lands on the good CDN
    for i in range(20):
        os.system(f"rm -f {QDIR}/tokenizer.json")
        os.system(f'curl -sSL -f {AUTH} "{BASE}/tokenizer.json" -o {QDIR}/tokenizer.json')
        if os.path.exists(f"{QDIR}/tokenizer.json") and os.path.getsize(f"{QDIR}/tokenizer.json") > 11_000_000:
            print(f"tokenizer.json ok on try {i+1}"); break
    assert ready(QDIR), "Qwen3 incomplete — re-run this cell (aria2c may need another pass)"
    os.makedirs(os.path.dirname(DRIVE_QWEN), exist_ok=True)
    shutil.copytree(QDIR, DRIVE_QWEN, dirs_exist_ok=True); print("cached Qwen3 to Drive")

# load by LOCAL PATH — never by hub name again
from sentence_transformers import SentenceTransformer
_m = SentenceTransformer(QDIR)
print("Qwen3 loaded offline:", _m.encode(["test"], normalize_embeddings=True).shape)  # (1, 1024)
del _m

## 4. Paths + ISOLATED data root (copy shared inputs from Drive `data/`)

In [ ]:
import os, shutil
# ISOLATED local root — Qwen3 stage2 is built here, never on Drive. Copied FRESH from Drive each run.
# Same structure as the v6rel run: {DRIVE}/data/{name}/{processed/stage1, raw/documents.json}
DRIVE_DATA = f"{DRIVE}/data"                 # shared inputs on Drive (read-only here)
DATA_ROOT  = "/content/data_v16sc"           # isolated local root
for name in ["tomato_train_v16sc", "tomato_test_v16sc"]:
    s1 = f"{DATA_ROOT}/{name}/processed/stage1"
    if os.path.exists(s1): shutil.rmtree(s1)             # REFRESH: re-copy every run (picks up re-uploads)
    os.makedirs(f"{DATA_ROOT}/{name}/processed", exist_ok=True)
    shutil.copytree(f"{DRIVE_DATA}/{name}/processed/stage1", s1)
    os.makedirs(f"{DATA_ROOT}/{name}/raw", exist_ok=True)
    shutil.copy(f"{DRIVE_DATA}/{name}/raw/documents.json", f"{DATA_ROOT}/{name}/raw/documents.json")
# benchmark baselines for the stratified comparison (BGE retrieval + v6rel KL-loss trained)
os.makedirs(f"{DATA_ROOT}/benchmark", exist_ok=True)
for f in ["predictions_bge.json", "predictions_tomato_test_v6rel_kl-loss.json"]:
    shutil.copy(f"{DRIVE_DATA}/benchmark/{f}", f"{DATA_ROOT}/benchmark/{f}")
print("DATA_ROOT (isolated, local):", DATA_ROOT)
!ls -R {DATA_ROOT} | head -50

# operator raw ingredients (dense/S/M/total_S, float16), aligned to each split's nodes.csv doc order.
# The operator is recomputed live from these so its scalars w, beta train jointly.
for name in ['tomato_train_v16sc', 'tomato_test_v16sc']:
    src = f'{DRIVE_DATA}/{name}/operator_components.npz'
    assert os.path.exists(src), f'MISSING {src} — upload operator_components.npz for {name} to Drive'
    shutil.copy(src, f'{DATA_ROOT}/{name}/operator_components.npz')
print('operator components copied')


## 5. Structural audit (must pass before training)

In [ ]:
import csv, json
for split in ["train", "test"]:
    s1 = f"{DATA_ROOT}/tomato_{split}_v16sc/processed/stage1"
    names = {r["name"] for r in csv.DictReader(open(f"{s1}/nodes.csv"))}
    edges = list(csv.DictReader(open(f"{s1}/edges.csv")))
    rels  = {r["name"] for r in csv.DictReader(open(f"{s1}/relations.csv"))}
    q = json.load(open(f"{s1}/{split}.json"))
    assert all(e["source"] in names and e["target"] in names for e in edges), "dangling edge endpoint"
    assert {e["relation"] for e in edges} <= rels, "edge relation not in relations.csv"
    bad = sum(any(n not in names for v in x["start_nodes"].values() for n in v) for x in q)
    seedless = sum(not any(x["start_nodes"].values()) for x in q)
    print(f"{split}: nodes={len(names):,} edges={len(edges):,} rels={len(rels)} queries={len(q):,} "
          f"unresolved_seeds={bad} seedless={seedless}")
print("structural checks passed")

## 6. Matched run function (same hydra entry as v6rel/v15)

In [ ]:
import os, subprocess
OUT_ROOT = f"{DRIVE}/outputs/v16sc"; os.makedirs(OUT_ROOT, exist_ok=True)
def run_model(epochs, force_reload, suffix):
    run_dir = f"{OUT_ROOT}/v16sc_{suffix}"; os.makedirs(run_dir, exist_ok=True)
    cmd = ["python","-u","-m","gfmrag.workflow.sft_training",
        "--config-path","config/gfm_reasoner","--config-name","sft_training_fusion",
        "text_emb_model=qwen3_st", f"datasets.cfgs.root={DATA_ROOT}",
        f"datasets.cfgs.force_reload={str(force_reload)}",
        "datasets.train_names=[tomato_train_v16sc]",
        "datasets.valid_names=[tomato_test_v16sc]",
        f"trainer.args.num_epoch={epochs}","trainer.args.train_batch_size=4",
        "+trainer.args.do_predict=true","+trainer.args.predict_top_k=100",
        f"hydra.run.dir={run_dir}"]
    env = dict(os.environ, WANDB_MODE="disabled", HYDRA_FULL_ERROR="1",
               PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True",
               STRAT_EVAL="1", STRAT_BGE=f"{DATA_ROOT}/benchmark/predictions_bge.json",
               STRAT_NAME="tomato_test_v16sc",
               STRAT_TEST=f"{DATA_ROOT}/tomato_test_v16sc/processed/stage1/test.json",
               OPERATOR_COMPONENTS=f"{DATA_ROOT}/tomato_train_v16sc/operator_components.npz",
               OPERATOR_COMPONENTS_TEST=f"{DATA_ROOT}/tomato_test_v16sc/operator_components.npz",
               FUSION_OBJECTIVE="hardneg",  # operator-hard-negative contrastive (report Eq 3.9)
               HARDNEG_HUB="50", HARDNEG_RAND="50",
               AUX_W="1.0")               # graph-alone hard-neg term: trains the GNN directly (survives gamma~0)
    with open(f"{run_dir}/console.log","w") as log:
        p = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in p.stdout: print(line, end=""); log.write(line); log.flush()
        assert p.wait() == 0, "training failed"
    return run_dir


## 7. Smoke forward pass (epochs=0, builds index)

In [ ]:
SMOKE = run_model(epochs=0, force_reload=True, suffix="fusion_smoke")


## 8. Full training (20 epochs, matches v6rel)

Each epoch's eval now logs **stratified** metrics too (patched in cell 2), e.g.
`tomato_test_v16sc/document_hits@5/cross+dissim: ...` alongside the aggregate
`tomato_test_v16sc/document_hits@5: ...`. Slices: `same`, `cross`, `sim`, `dissim`, `cross+dissim`.
A forward hook records gold-doc ranks **during the existing eval pass — no extra forward pass**, so
per-epoch cost is essentially unchanged. Training speed is unaffected. (Single-GPU: the hook sees all
eval queries; on multi-GPU it would see only the local shard.) Set `STRAT_EVAL=0` to disable; best-model
selection still uses aggregate `document_mrr`.

In [ ]:
RUN_DIR = run_model(epochs=20, force_reload=False, suffix="fusion_epoch20")
print("done ->", RUN_DIR)


## 9. Stratified evaluation (trainer only logs aggregate)

The gfmrag trainer logs one aggregate `document_hits@k` / `document_mrr`. This cell recomputes the SAME
metric names from the saved `predictions_*.json` (top-100) but **broken out by slice**
(same / cross by stratum, similar / dissimilar by BGE gold-rank ≤/>100), for the trained v16 run AND
the trained v6rel baseline, side by side. (mrr is capped at rank 100 = the saved depth; matches the
trainer's document_mrr to <0.01.)

In [ ]:
import json, os
from collections import defaultdict
KS = (1, 2, 3, 5, 10, 20, 50, 100)
bge = {r["id"]: r for r in json.load(open(f"{DATA_ROOT}/benchmark/predictions_bge.json"))}
def bge_rank(qid, g):
    for i,(d,_) in enumerate(bge.get(qid,{}).get("predictions",{}).get("document",[]),1):
        if d==g: return i
    return 10**9

def stratify(pred):
    sl = defaultdict(lambda: defaultdict(list))
    for r in pred:
        qid = r["id"]
        gold = r.get("supporting_documents") or r.get("target_nodes",{}).get("document",[])
        gd = gold[0] if isinstance(gold,list) and gold else gold
        ranked = [d for d,_ in r["predictions"]["document"]]
        rk = ranked.index(gd)+1 if gd in ranked else 10**9
        st = r.get("stratum","?"); sim = "dissimilar" if bge_rank(qid,gd) > 100 else "similar"
        for nm in ("all", st, sim, f"{st}+{sim}"):
            m = sl[nm]
            for k in KS: m[f"hits@{k}"].append(float(rk <= k))
            m["mrr"].append(1.0/rk if rk <= 100 else 0.0)
    return sl

def show(name, sl):
    order = ("all","same","cross","similar","dissimilar","same+dissimilar","cross+dissimilar")
    cols = ["hits@1","hits@5","hits@10","hits@20","mrr"]
    print(f"\n### {name}")
    print("| slice | n | " + " | ".join(f"document_{c}" for c in cols) + " |")
    print("|---|--:|" + "--:|"*len(cols))
    for nm in order:
        m = sl.get(nm)
        if not m: continue
        n = len(m["mrr"])
        print(f"| {nm} | {n} | " + " | ".join(f"{sum(m[c])/n:.4f}" for c in cols) + " |")

v16 = stratify(json.load(open(f"{RUN_DIR}/predictions_tomato_test_v16sc.json")))
show("v16sc soft+canon0.85 — trained G-Reasoner", v16)
base = f"{DATA_ROOT}/benchmark/predictions_tomato_test_v6rel_kl-loss.json"
if os.path.exists(base):
    show("v6rel KL-loss — trained baseline", stratify(json.load(open(base))))

# aggregate in the trainer's exact log format (sanity vs the console during training)
m = v16["all"]; n = len(m["mrr"])
print("\n# aggregate (trainer-format, compare to the epoch log):")
for c in ["mrr"] + [f"hits@{k}" for k in KS]:
    print(f"tomato_test_v16sc/document_{c}: {sum(m[c])/n:.4f}")

## Read

- **If cross+dissim R@5 clears the untrained-PPR 9.4 and beats trained v6rel 6.25**, the typed multi-hop frame paths pay off under learned routing — the hypothesis the untyped PPR could not test.
- **If same/all stays well below v6rel 38.3**, the long path lengths (median 3–4 hops) are bottlenecking a shallow reasoner — raise message-passing depth and/or tighten consolidation to shorten paths.
- Freeze the construction and re-run on an untouched split before any final claim.

## 10. MRR trajectory (aggregate + per-slice, per epoch)

In [ ]:
# === MRR trajectory (aggregate + per-slice) from the training log ===
import re
LOG  = f"{RUN_DIR}/console.log"          # RUN_DIR = the 20-epoch run
NAME = "tomato_test_v16sc"
txt  = open(LOG).read()
series = lambda pat: [float(x) for x in re.findall(pat, txt)]

agg    = series(rf"{NAME}/document_mrr:\s*([0-9.]+)")           # aggregate MRR, one per eval
slices = ["same", "cross", "sim", "dissim", "cross+dissim"]
strat  = {s: series(rf"{NAME}/document_mrr/{re.escape(s)}:\s*([0-9.]+)") for s in slices}

n = len(agg)
print(f"{n} evaluations logged (epochs 1..{max(n-1,0)} + final)\n")
hdr = f"{'ep':>3} | {'agg':>6} | " + " | ".join(f"{s:>12}" for s in slices)
print(hdr); print("-" * len(hdr))
for i in range(n):
    cells = " | ".join(f"{strat[s][i]:>12.3f}" if i < len(strat[s]) else f"{chr(45):>12}" for s in slices)
    print(f"{i:>3} | {agg[i]:>6.3f} | {cells}")

if agg:
    b = max(range(n), key=lambda i: agg[i])
    print(f"\nbest aggregate MRR = {agg[b]:.3f} at eval {b}")
    for s in slices:
        if strat[s]:
            bs = max(range(len(strat[s])), key=lambda i: strat[s][i])
            print(f"  best {s:>12} MRR = {strat[s][bs]:.3f} at eval {bs}")

try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(9, 5))
    plt.plot(agg, "ko-", lw=2, label="aggregate")
    for s in slices:
        if strat[s]: plt.plot(strat[s], "o--", label=s)
    plt.xlabel("evaluation (\u2248 epoch)"); plt.ylabel("document_mrr")
    plt.title("v16sc MRR trajectory"); plt.legend(); plt.grid(alpha=0.3); plt.show()
except Exception as e:
    print("plot skipped:", e)


## 12. Qwen-operator (optional): rebuild the operator on **Qwen3** embeddings

Encodes docs / queries / probes with the same Qwen3 already loaded, recomputes the operator's
`dense / S / M`, applies the fitted formula, and scores the master-table slices — to compare a
**Qwen-operator** against the BGE-operator (aggregate gain vs the unchanged dissimilar blind spot).
Standalone (no training needed). **Needs one upload:** `probes_test.jsonl` (~6 MB) at
`{DRIVE}/data/probes_test.jsonl` (it's `kg-construction/construct_v2/cache/probes_test.jsonl`).


In [ ]:

# === Qwen-operator: the operator formula on Qwen3 embeddings, vs the BGE-operator ===
import numpy as np, json, csv, os
from collections import defaultdict
from sentence_transformers import SentenceTransformer

# same custom retrieval instruction as the G-Reasoner's Qwen3 (query side only)
QINSTR = ("Instruct: Given a scientific research problem or open need, retrieve papers whose method, "
          "mechanism, or technique could be borrowed as inspiration, including transfers from other "
          "domains.\nQuery: ")
W, B = [1.05, 1.05, 0.25], 0.95
z = lambda X: (X - X.mean(1, keepdims=True)) / (X.std(1, keepdims=True) + 1e-6)

qm = SentenceTransformer("/content/qwen3")
def enc(texts, instr=None, bs=64):
    t = [instr + x for x in texts] if instr else texts
    return np.asarray(qm.encode(t, normalize_embeddings=True, batch_size=bs, show_progress_bar=True),
                      dtype=np.float32)

s1 = f"{DATA_ROOT}/tomato_test_v16sc/processed/stage1"
docs = json.load(open(f"{DATA_ROOT}/tomato_test_v16sc/raw/documents.json"))      # {doc_id: text}
docnodes = [r["name"] for r in csv.DictReader(open(f"{s1}/nodes.csv")) if r["type"] == "document"]
test = json.load(open(f"{s1}/test.json"))
qids = [q["id"] for q in test]

PROBES = f"{DRIVE}/data/probes_test.jsonl"
assert os.path.exists(PROBES), f"upload probes_test.jsonl to {PROBES} (~6MB) — see the header cell"
probes = {json.loads(l)["id"]: json.loads(l)["probes"] for l in open(PROBES)}

print("encoding docs / queries / probes with Qwen3 ...")
de = enc([docs[d] for d in docnodes])                # passage side, nodes.csv doc order
qe = enc([q["question"] for q in test], instr=QINSTR)
flat, own = [], []
for q in qids:
    for p in probes.get(q, []): flat.append(p); own.append(q)
pe = enc(flat); own = np.array(own)

D, Q = len(docnodes), len(qids)
dense = qe @ de.T
S = np.zeros((Q, D), np.float32); M = np.zeros((Q, D), np.float32)
for i, q in enumerate(qids):
    idx = np.where(own == q)[0]
    if len(idx):
        H = np.clip(pe[idx] @ de.T, 0, None); S[i] = H.sum(0); M[i] = H.max(0)
dem = np.clip(S.sum(0, keepdims=True) - S, 1e-6, None)
Sop = W[0]*z(dense) + W[1]*z(S/dem**B) + W[2]*z(M/dem**B)

di = {d: i for i, d in enumerate(docnodes)}
meta = {q["id"]: q for q in test}
bge = {r["id"]: r for r in json.load(open(f"{DATA_ROOT}/benchmark/predictions_bge.json"))}
def brank(qid, g):
    for i, (d, _) in enumerate(bge.get(qid, {}).get("predictions", {}).get("document", []), 1):
        if d == g: return i
    return 10**9

sl = defaultdict(lambda: defaultdict(list)); KS = (1, 5, 10, 20)
for r, qid in enumerate(qids):
    g = meta[qid]["supporting_documents"][0]
    order = np.argsort(-Sop[r]); rk = int(np.where(order == di[g])[0][0]) + 1
    st = "same" if meta[qid].get("stratum") == "same" else "cross"
    sim = "dissimilar" if brank(qid, g) > 100 else "similar"
    for nm in ["all", st, sim] + (["cross+dissim"] if st == "cross" and sim == "dissimilar" else []):
        for k in KS: sl[nm][f"R@{k}"].append(float(rk <= k))

print("\nQwen-operator (Qwen3 embeddings, fitted w/β) — per-gold, test")
print(f"{'slice':13} {'n':>5}  " + "  ".join(f"R@{k}" for k in KS))
for nm in ("all", "same", "cross", "similar", "dissimilar", "cross+dissim"):
    m = sl[nm]; n = len(m["R@1"])
    print(f"{nm:13} {n:>5}  " + "  ".join(f"{100*np.mean(m[f'R@{k}']):4.1f}" for k in KS))
print("\nBGE-operator reference:  all 22.1/42.0 | cross 14.5/30.4 | dissimilar 0.2/1.9 | cross+dissim 0.6/5.0")
